***
#  📊 Raw Data Processing | Meat Consumption Project
This notebook applies all cleaning and transformation steps to the raw datasets.
The goal is to produce a single, clean dataset ready for training.
***
**Index**

* [Initial set up](#initial-set-up)

* [Data processing](#data-processing)

    - [FAO – Food Balance Sheets dataset](#fao-food-balance-sheets-dataset)
    - [World Bank – GDP per capita](#world-bank-gdp-per-capita)
    - [World Bank – Urban Population (%)](#world-bank--urban-population-)
     - [OWID – Global Meat Production](#owid-global-meat-production)
* [Merging Datasets](#merging-datasets)

## Initial set up

In [1]:
%load_ext autoreload
%autoreload 2

In [5]:
import sys
sys.path.append('src')  #path to src
import pandas as pd
from data_loader import *
import data_preprocessing as dp

## Data processing

In this section, we clean and harmonize the raw datasets and merge them into a single DataFrame suitable for training. This includes:

- Filtering and renaming key columns
- Selecting overlapping years across datasets
- Aligning countries and handling inconsistencies
- Merging all features into a single dataframe

At the end of each cleaning section we have the filtered datased ready to merge which is exported as `meat_processed_merged_data.csv`.


In [104]:
# Loading all raw datasets from '/Data/raw' using our module load_data.
# The module is written to acces directly the folder that contains the raw data, which means we do not need to specify paths to the file

dfs = load_raw_data()

# Identifying the 'keys' for each loaded file.
list(dfs.keys())

['faostat', 'gdp', 'urban', 'education', 'enviroment', 'production']

Once we load all data, we will proceed to make the initial exploration of every file to have information about

- size
- Features
- Null values

In order to simplify our cleaning & reorganization of datasets, we will rename our loaded data frames localy as follows:

- fao = dfs[`'faoastat'`] 
- gdp = dfs[`'gdp'`]
- urban = dfs[`'urban'`]
- edu = dfs[`'education'`]
- env = dfs=[`'enviroment'`]
- prod = [`'production'`]


### FAO – Food Balance Sheets dataset

In [52]:
# Local name for dfs.['faoastat']
fao = dfs['faostat']

***
Initial Exploration
***

In [53]:
# We obtain the general information of fao dataset
print(fao.info())
print(f'The file contains {fao.shape[0]}  rows & {fao.shape[1]} columns ')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11865 entries, 0 to 11864
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Domain Code       11865 non-null  object 
 1   Domain            11865 non-null  object 
 2   Area Code (M49)   11865 non-null  int64  
 3   Area              11865 non-null  object 
 4   Element Code      11865 non-null  int64  
 5   Element           11865 non-null  object 
 6   Item Code (FBS)   11865 non-null  object 
 7   Item              11865 non-null  object 
 8   Year Code         11865 non-null  int64  
 9   Year              11865 non-null  int64  
 10  Unit              11865 non-null  object 
 11  Value             11865 non-null  float64
 12  Flag              11865 non-null  object 
 13  Flag Description  11865 non-null  object 
 14  Note              0 non-null      float64
dtypes: float64(2), int64(4), object(9)
memory usage: 1.4+ MB
None
The file contains 11865  

In [54]:
# Information of the first 3 rows in our dataset, to se what kind of information is included
fao.tail(3)

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (FBS),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
11862,FBS,Food Balances (2010-),716,Zimbabwe,645,Food supply quantity (kg/capita/yr),S2735,"Meat, Other",2020,2020,kg/cap,2.35,E,Estimated value,NaN
11863,FBS,Food Balances (2010-),716,Zimbabwe,645,Food supply quantity (kg/capita/yr),S2735,"Meat, Other",2021,2021,kg/cap,2.35,E,Estimated value,NaN
11864,FBS,Food Balances (2010-),716,Zimbabwe,645,Food supply quantity (kg/capita/yr),S2735,"Meat, Other",2022,2022,kg/cap,2.35,E,Estimated value,NaN


In [55]:
#Checking if there is any null quantity across our dataset
fao.isnull().sum()

Domain Code             0
Domain                  0
Area Code (M49)         0
Area                    0
Element Code            0
Element                 0
Item Code (FBS)         0
Item                    0
Year Code               0
Year                    0
Unit                    0
Value                   0
Flag                    0
Flag Description        0
Note                11865
dtype: int64

***
**Filtering.** The main features are `'Area'`, `'Item'`, `'Year'` & `'Value'`, therefore we can drop all other columns. Since we need just a few columns we can define the `fao_filtered` new dataset containing only the columns of interest.
***

In [56]:
fao_filtered = fao[['Area','Year','Item','Value']]
#We will keep using this from now on.
fao_filtered


,Area,Year,Item,Value
0,Afghanistan,2010,Bovine Meat,4.74
1,Afghanistan,2011,Bovine Meat,4.80
2,Afghanistan,2012,Bovine Meat,4.40
3,Afghanistan,2013,Bovine Meat,4.18
4,Afghanistan,2014,Bovine Meat,4.99
...,...,...,...,...
11860,Zimbabwe,2018,"Meat, Other",2.47
11861,Zimbabwe,2019,"Meat, Other",2.45
11862,Zimbabwe,2020,"Meat, Other",2.35
11863,Zimbabwe,2021,"Meat, Other",2.35


***
General exploration to understand which kind of content we have in our dataset.
***

In [57]:
# We use uniqe to list all kinds of meats
print('Type of meat reported:', fao_filtered['Item'].unique())

# We use nunique to identify the nomber of countries the data is coming from
print('Countries reported:', fao_filtered['Area'].nunique())

# We use max() and min() to know the period of time when the data was collected
print('Year range:',fao_filtered['Year'].min(), '-' , fao_filtered['Year'].max())


Type of meat reported: ['Bovine Meat' 'Mutton & Goat Meat' 'Pigmeat' 'Poultry Meat' 'Meat, Other']
Countries reported: 190
Year range: 2010 - 2022


In [58]:
# Checking null values in the new filtered data set
print(fao_filtered.info())
print('---------------------------------------')
print('Null numbers in each column:')
print(fao_filtered.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11865 entries, 0 to 11864
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Area    11865 non-null  object 
 1   Year    11865 non-null  int64  
 2   Item    11865 non-null  object 
 3   Value   11865 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 370.9+ KB
None
---------------------------------------
Null numbers in each column:
Area     0
Year     0
Item     0
Value    0
dtype: int64


In [59]:
# We drop all the dataset corresponding to 'Meat,Other'
fao_filtered = fao_filtered[fao_filtered['Item'] != 'Meat, Other']
fao_filtered 


,Area,Year,Item,Value
0,Afghanistan,2010,Bovine Meat,4.74
1,Afghanistan,2011,Bovine Meat,4.80
2,Afghanistan,2012,Bovine Meat,4.40
3,Afghanistan,2013,Bovine Meat,4.18
4,Afghanistan,2014,Bovine Meat,4.99
...,...,...,...,...
11847,Zimbabwe,2018,Poultry Meat,4.38
11848,Zimbabwe,2019,Poultry Meat,4.47
11849,Zimbabwe,2020,Poultry Meat,7.21
11850,Zimbabwe,2021,Poultry Meat,7.33


***
**Renaming & Transforming columns.** We rename the columns `Area` and `Value` to `Country` and `meat_consumption`. Then, we convert each type of meat into a column. Each meat type will be assigned the value of meat consuption (kg/capita/yr).
***

In [60]:

# 1. Rename columns we use rename(.,.) and assign the neu names to the filtered dataset by doung fao_filtered = fao_filtered.rename(.,.)
fao_filtered = fao_filtered.rename(columns={'Area':'Country','Value':'meat_consumption'})

In [61]:
# 2. Convert  meat types into columns: Pivot the filtered data and reasign it. 
# index: Columns that will stay fixed
# columns: To convert to new columns
# Values: Values that will be in each column
#------------------------------------------------------------------------------------------------------------------------
# IMPORTANT: Run this ONLY once, or it will create an error, since we are re asigning the new shape to our filtered data 
#------------------------------------------------------------------------------------------------------------------------

fao_filtered = fao_filtered.pivot_table(
    index   = ['Country', 'Year'],
    columns = 'Item',  
    values  = 'meat_consumption', 
    aggfunc = 'sum' # In case of duplicates
    
).reset_index()


In [62]:
# Checking the final form ofthis dataset
# use head() or tail() to explore more
fao_filtered.tail(3)

Item,Country,Year,Bovine Meat,Mutton & Goat Meat,Pigmeat,Poultry Meat
2380,Zimbabwe,2020,40.12,1.76,0.64,7.21
2381,Zimbabwe,2021,43.77,1.95,0.73,7.33
2382,Zimbabwe,2022,44.42,1.96,0.69,7.18


In [63]:
# Checking null values in the new filtered data set
print(fao_filtered.info())
print('---------------------------------------')
print('Null numbers in each column:')
print(fao_filtered.isnull().sum())
print(f'The file contains {fao_filtered.shape[0]}  rows & {fao_filtered.shape[1]} columns ')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2383 entries, 0 to 2382
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country             2383 non-null   object 
 1   Year                2383 non-null   int64  
 2   Bovine Meat         2382 non-null   float64
 3   Mutton & Goat Meat  2383 non-null   float64
 4   Pigmeat             2337 non-null   float64
 5   Poultry Meat        2383 non-null   float64
dtypes: float64(4), int64(1), object(1)
memory usage: 111.8+ KB
None
---------------------------------------
Null numbers in each column:
Item
Country                0
Year                   0
Bovine Meat            1
Mutton & Goat Meat     0
Pigmeat               46
Poultry Meat           0
dtype: int64
The file contains 2383  rows & 6 columns 


### World Bank – GDP per capita

In [64]:
#Local name for dfs['gdp']
gdp = dfs['gdp']

In [65]:
gdp["Indicator Name"].value_counts()

Indicator Name
GDP per capita (current US$)    266
Name: count, dtype: int64

***
Initial exploration
***

In [66]:
print(gdp.info())
print(f'The file contains {gdp.shape[0]}  rows & {gdp.shape[1]} columns ')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 70 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    266 non-null    object 
 1   Country Code    266 non-null    object 
 2   Indicator Name  266 non-null    object 
 3   Indicator Code  266 non-null    object 
 4   1960            151 non-null    float64
 5   1961            154 non-null    float64
 6   1962            156 non-null    float64
 7   1963            156 non-null    float64
 8   1964            156 non-null    float64
 9   1965            162 non-null    float64
 10  1966            163 non-null    float64
 11  1967            167 non-null    float64
 12  1968            168 non-null    float64
 13  1969            168 non-null    float64
 14  1970            190 non-null    float64
 15  1971            191 non-null    float64
 16  1972            191 non-null    float64
 17  1973            191 non-null    flo

In [67]:
# Initial exploration of the gdp dataset
print('Null numbers in each column:')
print(gdp.isnull().sum())   
gdp.head(2)

Null numbers in each column:
Country Name        0
Country Code        0
Indicator Name      0
Indicator Code      0
1960              115
                 ... 
2021                8
2022                9
2023               17
2024               34
Unnamed: 69       266
Length: 70, dtype: int64


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,Unnamed: 69
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,27441.529662,28440.051964,30082.127645,31096.205074,22855.93232,27200.061079,30559.533535,33984.790620,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.121835,186.941781,197.402402,225.440494,208.999748,226.876513,...,1329.807285,1520.212231,1538.901679,1493.817938,1344.10321,1522.393346,1628.318944,1568.159891,1673.841139,NaN


***
**Important!** Before filtering, we first need to manipulate this data: 
- Keep column  `'Country Name'`
- create columns `'Year'` & `'GDP_per_capita'`
***

In [68]:
#We keep 'Country Name' and create new columns 'Year' & 'GDP_per_capita
# We will create a new column 'Year' with the years from 2010 to 2022
# We will create a new column 'GDP_per_capita' with the GDP per capita values for each year

#New columns for gdp
new_columns_gdp = ['Country Name']+[str(year) for year in range(2010,2023)]
#Filtering the urban dataset to keep only the 'Urban population (% of total population)' indicator
# This is done to focus on the relevant data for our analysi
#gdp= gdp[gdp['Indicator Name'] == 'GDP per capita (current US$)']

In [69]:
# Melting the required data from 2010 to 2022
gdp_melted = gdp[new_columns_gdp ].melt(id_vars='Country Name', var_name='Year', value_name='GDP_per_capita')


In [70]:
# Checking the new shape of the melted gdp dataset
gdp_melted.tail(3)

,Country Name,Year,GDP_per_capita
3455,South Africa,2022,6523.410978
3456,Zambia,2022,1447.123101
3457,Zimbabwe,2022,2040.546587


In [71]:
#Checking general information of the melted gdp dataset
print('---------------------------------------')
print('General information of the melted gdp dataset:')
print(gdp_melted.info())
print('---------------------------------------')
print(f'The file contains {gdp_melted.shape[0]}  rows & {gdp_melted.shape[1]} columns ')
print('Null numbers in each column:')
print(gdp_melted.isnull().sum())

---------------------------------------
General information of the melted gdp dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3458 entries, 0 to 3457
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    3458 non-null   object 
 1   Year            3458 non-null   object 
 2   GDP_per_capita  3369 non-null   float64
dtypes: float64(1), object(2)
memory usage: 81.2+ KB
None
---------------------------------------
The file contains 3458  rows & 3 columns 
Null numbers in each column:
Country Name       0
Year               0
GDP_per_capita    89
dtype: int64


***
**Notice** `'Year'` column is given as an object.
***

In [72]:
# In order to change 'Year' as an object to an integer, we need to convert it first
gdp_melted['Year'] = gdp_melted['Year'].astype(int)

In [73]:
# Now we can check the data type of 'Year'
print(gdp_melted['Year'].dtype)

int64


In [74]:
#For standarization we just redefine dataset name and rename Country Name as Country
gdp_filtered=gdp_melted
gdp_filtered = gdp_filtered.rename(columns={'Country Name': 'Country'})
gdp_filtered

,Country,Year,GDP_per_capita
0,Aruba,2010,24093.140151
1,Africa Eastern and Southern,2010,1601.727651
2,Afghanistan,2010,560.621505
3,Africa Western and Central,2010,1663.966937
4,Angola,2010,3597.342932
...,...,...,...
3453,Kosovo,2022,5290.947472
3454,"Yemen, Rep.",2022,615.702078
3455,South Africa,2022,6523.410978
3456,Zambia,2022,1447.123101


In [75]:
gdp_filtered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3458 entries, 0 to 3457
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country         3458 non-null   object 
 1   Year            3458 non-null   int64  
 2   GDP_per_capita  3369 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 81.2+ KB


In [76]:
gdp_melted.groupby(['Country Name', 'Year']).size().value_counts()



1    3458
Name: count, dtype: int64

### World Bank – Urban Population (%)

In [77]:
# Local name for dfs['urban']
urban = dfs['urban']

In [78]:
# Checking the general formation of the urban dataset
urban.tail(3)

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,Unnamed: 69
263,South Africa,ZAF,Urban population (% of total population),SP.URB.TOTL.IN.ZS,46.619,46.793,46.906,47.020,47.134,47.248,...,65.341,65.850,66.355,66.856,67.354,67.847,68.335,68.819,69.298,NaN
264,Zambia,ZMB,Urban population (% of total population),SP.URB.TOTL.IN.ZS,18.145,18.951,19.785,20.712,22.015,23.372,...,42.438,42.976,43.521,44.072,44.629,45.192,45.761,46.335,46.914,NaN
265,Zimbabwe,ZWE,Urban population (% of total population),SP.URB.TOTL.IN.ZS,12.608,12.821,13.082,13.578,14.092,14.620,...,32.296,32.237,32.209,32.210,32.242,32.303,32.395,32.517,32.670,NaN


The urban dataset is of the exact form as the gdp dataset. Therefore, for quick performance we will do all necessary changes in one cell.

In [79]:
# Actions needed for the urban dataset
#New columns for gdp
new_columns_urban = ['Country Name']+[str(year) for year in range(2010,2023)]
#Filtering the urban dataset to keep only the 'Urban population (% of total population)' indicator
# This is done to focus on the relevant data for our analysi
#urban = urban[urban['Indicator Name'] == 'Urban population (% of total population)']
urban_filtered=urban[new_columns_urban ].melt(id_vars='Country Name', var_name='Year', value_name='Urban_population')
urban_filtered['Year'] = urban_filtered['Year'].astype(int)

#For standarization we just redefine dataset name and rename Country Name as Country
urban_filtered = urban_filtered.rename(columns={'Country Name': 'Country'})


In [80]:
# Checking output
urban_filtered

,Country,Year,Urban_population
0,Aruba,2010,43.059000
1,Africa Eastern and Southern,2010,32.195595
2,Afghanistan,2010,23.737000
3,Africa Western and Central,2010,41.672423
4,Angola,2010,59.783000
...,...,...,...
3453,Kosovo,2022,NaN
3454,"Yemen, Rep.",2022,39.188000
3455,South Africa,2022,68.335000
3456,Zambia,2022,45.761000


In [81]:
# General information of the filtered urban dataset 
print(urban_filtered.info())
print('---------------------------------------')
print('Null numbers in each column:')
print(urban_filtered.isnull().sum())
print(f'The file contains {urban_filtered.shape[0]}  rows & {urban_filtered.shape[1]} columns ')    

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3458 entries, 0 to 3457
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Country           3458 non-null   object 
 1   Year              3458 non-null   int64  
 2   Urban_population  3419 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 81.2+ KB
None
---------------------------------------
Null numbers in each column:
Country              0
Year                 0
Urban_population    39
dtype: int64
The file contains 3458  rows & 3 columns 


In [82]:
gdp_melted.groupby(['Country Name', 'Year']).size().value_counts()

1    3458
Name: count, dtype: int64

###  OWID – Global Meat Production

In [83]:
# Local name for dfs['production']  
prod= dfs['production']

In [84]:
# Checkimg the general formation of the production dataset
prod.head(3)

,Entity,Code,Year,"Meat, total | 00001765 || Production | 005510 || tonnes"
0,Afghanistan,AFG,1961,129420.00
1,Afghanistan,AFG,1962,132205.73
2,Afghanistan,AFG,1963,138971.36


In [85]:
# Initial exploration
print(prod.info())
print(f'The file contains {prod.shape[0]}  rows & {prod.shape[1]} columns ')
# Checking the general formation of the production dataset
print('---------------------------------------')
print('Null numbers in each column:')
print(prod.isnull().sum())  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14614 entries, 0 to 14613
Data columns (total 4 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   Entity                                                   14614 non-null  object 
 1   Code                                                     11909 non-null  object 
 2   Year                                                     14614 non-null  int64  
 3   Meat, total | 00001765 || Production | 005510 || tonnes  14614 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 456.8+ KB
None
The file contains 14614  rows & 4 columns 
---------------------------------------
Null numbers in each column:
Entity                                                        0
Code                                                       2705
Year                                                          0
Mea

***
**Filtering data** We drop the column `'Code'` and keep the rest.
***

In [86]:
# Calling relevant columns & filtering the dataset.
prod_filtered = prod[['Entity','Year','Meat, total | 00001765 || Production | 005510 || tonnes']]

# Checking the filtered production dataset
prod_filtered

,Entity,Year,"Meat, total | 00001765 || Production | 005510 || tonnes"
0,Afghanistan,1961,129420.00
1,Afghanistan,1962,132205.73
2,Afghanistan,1963,138971.36
3,Afghanistan,1964,143830.00
4,Afghanistan,1965,150195.00
...,...,...,...
14609,Zimbabwe,2019,823826.10
14610,Zimbabwe,2020,821114.40
14611,Zimbabwe,2021,895893.50
14612,Zimbabwe,2022,926426.20


In [87]:
print('Years in dataset: from', {prod_filtered['Year'].min()}, 'to', {prod_filtered['Year'].max()})
print('---------------------------------------')
print('Country in dataset:', prod_filtered['Entity'].nunique())

Years in dataset: from {np.int64(1961)} to {np.int64(2023)}
---------------------------------------
Country in dataset: 254


***
**Actions needed** 
- Rename columns 'Entity' & 'Meat, total | 00001765 || Production | 005510 || tonnes'
- Range of years 2010- - 2022
*** 

In [88]:
prod_filtered = prod_filtered.rename(columns={'Entity': 'Country', 'Meat, total | 00001765 || Production | 005510 || tonnes': 'Production'})
#prod_filtered = prod_filtered[(prod_filtered['Year'] >= 2010) & (prod_filtered['Year'] <= 2022)]
prod_filtered

,Country,Year,Production
0,Afghanistan,1961,129420.00
1,Afghanistan,1962,132205.73
2,Afghanistan,1963,138971.36
3,Afghanistan,1964,143830.00
4,Afghanistan,1965,150195.00
...,...,...,...
14609,Zimbabwe,2019,823826.10
14610,Zimbabwe,2020,821114.40
14611,Zimbabwe,2021,895893.50
14612,Zimbabwe,2022,926426.20


In [89]:
prod_filtered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14614 entries, 0 to 14613
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Country     14614 non-null  object 
 1   Year        14614 non-null  int64  
 2   Production  14614 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 342.6+ KB


## Merging datasets





In [90]:
# Applying  standard names to my dataset
fao_cleaned = dp.standardize_country_names(fao_filtered)
gdp_cleaned = dp.standardize_country_names(gdp_filtered)
urban_cleaned = dp.standardize_country_names(urban_filtered)
prod_cleaned= dp.standardize_country_names(prod_filtered)

More than one regular expression match for China, Taiwan Province of
More than one regular expression match for China, Taiwan Province of
More than one regular expression match for China, Taiwan Province of
More than one regular expression match for China, Taiwan Province of
More than one regular expression match for China, Taiwan Province of
More than one regular expression match for China, Taiwan Province of
More than one regular expression match for China, Taiwan Province of
More than one regular expression match for China, Taiwan Province of
More than one regular expression match for China, Taiwan Province of
More than one regular expression match for China, Taiwan Province of
More than one regular expression match for China, Taiwan Province of
More than one regular expression match for China, Taiwan Province of
More than one regular expression match for China, Taiwan Province of
More than one regular expression match for China, Taiwan Province of
More than one regular expression m

In [91]:


converted = pd.DataFrame({
    'original': fao_filtered['Country'],
    'converted': fao_cleaned['Country']
})
converted.head(25)

,original,converted
0,Afghanistan,Afghanistan
1,Afghanistan,Afghanistan
2,Afghanistan,Afghanistan
3,Afghanistan,Afghanistan
4,Afghanistan,Afghanistan
5,Afghanistan,Afghanistan
6,Afghanistan,Afghanistan
7,Afghanistan,Afghanistan
8,Afghanistan,Afghanistan
9,Afghanistan,Afghanistan


In [92]:
merge1 = dp.mergingfunc(fao_filtered,gdp_filtered)
merge2 = dp.mergingfunc(merge1, urban_filtered)
merge_df = dp.mergingfunc(merge2,prod_filtered)

In [99]:
# Checking the final shape of the merged dataset
print(merge_df.info())
print(merge_df.duplicated(subset=["Country", "Year"]).sum())
print(merge_df["Country"].value_counts().head())
print(merge_df["Year"].value_counts().sort_index())
merge_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1933 entries, 0 to 1932
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country             1933 non-null   object 
 1   Year                1933 non-null   int64  
 2   Bovine Meat         1932 non-null   float64
 3   Mutton & Goat Meat  1933 non-null   float64
 4   Pigmeat             1887 non-null   float64
 5   Poultry Meat        1933 non-null   float64
 6   GDP_per_capita      1927 non-null   float64
 7   Urban_population    1933 non-null   float64
 8   Production          1933 non-null   float64
dtypes: float64(7), int64(1), object(1)
memory usage: 136.0+ KB
None
0
Country
Afghanistan            13
Albania                13
Algeria                13
Angola                 13
Antigua and Barbuda    13
Name: count, dtype: int64
Year
2010    147
2011    147
2012    148
2013    148
2014    148
2015    148
2016    148
2017    148
2018    147
2019   

,Year,Bovine Meat,Mutton & Goat Meat,Pigmeat,Poultry Meat,GDP_per_capita,Urban_population,Production
count,1933.000000,1932.000000,1933.000000,1887.000000,1933.000000,1927.000000,1933.000000,1.933000e+03
mean,2016.030005,11.171082,3.268215,12.945760,20.285737,13834.521481,57.402320,1.640913e+06
std,3.744303,9.342133,5.802603,14.580272,16.563685,19832.439740,22.515159,7.328020e+06
min,2010.000000,0.000000,0.000000,0.000000,0.360000,210.008140,10.642000,7.779000e+01
25%,2013.000000,4.307500,0.560000,1.050000,6.330000,1835.441952,40.277000,7.052900e+04
50%,2016.000000,7.920000,1.460000,7.470000,17.930000,5879.337923,57.895000,2.655768e+05
75%,2019.000000,16.152500,3.780000,20.150000,28.950000,16233.213315,76.199000,8.775780e+05
max,2022.000000,55.180000,68.450000,67.600000,120.220000,134965.815442,100.000000,9.342856e+07


In [100]:
#Visualizing the final merged dataset
merge_df

,Country,Year,Bovine Meat,Mutton & Goat Meat,Pigmeat,Poultry Meat,GDP_per_capita,Urban_population,Production
0,Afghanistan,2010,4.74,5.04,NaN,2.29,560.621505,23.737,328160.0
1,Afghanistan,2011,4.80,4.66,NaN,1.92,606.694676,23.948,335664.0
2,Afghanistan,2012,4.40,5.03,NaN,2.05,651.417134,24.160,330600.0
3,Afghanistan,2013,4.18,5.20,NaN,2.11,637.087099,24.373,322110.0
4,Afghanistan,2014,4.99,5.06,0.03,2.11,625.054942,24.587,311019.7
...,...,...,...,...,...,...,...,...,...
1928,Zimbabwe,2018,40.99,1.82,0.52,4.38,2271.852504,32.209,771502.0
1929,Zimbabwe,2019,40.98,1.77,0.69,4.47,1683.913136,32.210,823826.1
1930,Zimbabwe,2020,40.12,1.76,0.64,7.21,1730.453910,32.242,821114.4
1931,Zimbabwe,2021,43.77,1.95,0.73,7.33,1724.387271,32.303,895893.5


In [101]:
#Final checks
print("✅ Shape:", merge_df.shape)
print("✅ Duplicates by Country/Year:", merge_df.duplicated(["Country", "Year"]).sum())
print("✅ Nulls:\n", merge_df.isna().sum())
print("✅ Year range:", merge_df["Year"].min(), "-", merge_df["Year"].max())


✅ Shape: (1933, 9)
✅ Duplicates by Country/Year: 0
✅ Nulls:
 Country                0
Year                   0
Bovine Meat            1
Mutton & Goat Meat     0
Pigmeat               46
Poultry Meat           0
GDP_per_capita         6
Urban_population       0
Production             0
dtype: int64
✅ Year range: 2010 - 2022


In [ ]:
save_path = '../data/processed/'
if not os.path.exists(save_path):
    os.makedirs(save_path)  
merge_df.to_csv(os.path.join(save_path, 'meat_processed_merged_data.csv'), index=False)

## Nan analisis

Checking the type of meat consuption in each country and if there is missing data in the master file

In [ ]:
master_file = pd.read_csv('data/raw/meat_processed_merged_data.csv')
print(master_file.isnull().sum())

Country                0
Year                   0
Bovine Meat            1
Mutton & Goat Meat     0
Pigmeat               46
Poultry Meat           0
GDP_per_capita         6
Urban_population       0
Production             0
dtype: int64


Some countries do not consume pig meat and only one country does not consume Bovine Meat (cow). 

In the following we analise if this is due to real 'missing' data or real zero consumption

In [8]:
# According to the general Nan results, check which countries have NaN values in the 'Pigmeat' and 'Bovine Meat' columns
pigmeat_nan_df = master_file[master_file["Pigmeat"].isna()]
cow_nan_df = master_file[master_file["Bovine Meat"].isna()]
# Print it as a list 
print('Countries twith Nan in pig meat:', list(pigmeat_nan_df['Country'].unique()) ) # List countries with NaN in Pigmeat column
print('Countries with Nan in cow meat:', list(cow_nan_df['Country'].unique()) )# List countries with NaN in Bovine Meat column

Countries twith Nan in pig meat: ['Afghanistan', 'Kuwait', 'Mauritania', 'Pakistan', 'Saudi Arabia', 'Tunisia', 'United Arab Emirates']
Countries with Nan in cow meat: ['Kiribati']


Countries with null meat consuption of pig (muslim countries), cow (not vialbe its production) $\to$ We safely set the Nans to zero values

In [9]:
# Create the list of countries with NaN values in pig meat and cow meat in our master dataset 
muslum_countries = list(pigmeat_nan_df['Country'].unique())
# These countries are Muslim countries, and so NaN indicates null consumption or restringered/negligible. We can safely set this to 0
master_file.loc[master_file['Country'].isin(muslum_countries) & (master_file['Pigmeat'].isnull()), 'Pigmeat'] = 0           
master_file.loc[master_file['Country'].isin(list(cow_nan_df['Country'].unique())) & (master_file['Bovine Meat'].isnull()), 'Bovine Meat'] = 0
print(master_file.isnull().sum())

Country               0
Year                  0
Bovine Meat           0
Mutton & Goat Meat    0
Pigmeat               0
Poultry Meat          0
GDP_per_capita        6
Urban_population      0
Production            0
dtype: int64


In [10]:
gdp_per_capita =master_file[master_file['GDP_per_capita'].isna()]
print('Countries with NaN in GDP per capita:', list(gdp_per_capita['Country'].unique()) ) # List countries with NaN in GDP per capita column
gdp_per_capita

Countries with NaN in GDP per capita: ['Cuba', 'South Sudan']


,Country,Year,Bovine Meat,Mutton & Goat Meat,Pigmeat,Poultry Meat,GDP_per_capita,Urban_population,Production
440,Cuba,2021,7.90,2.18,17.25,32.45,NaN,77.292,257217.00
441,Cuba,2022,8.09,1.68,18.81,29.80,NaN,77.401,315534.97
1663,South Sudan,2019,13.16,4.80,0.01,8.41,NaN,19.899,295223.00
1664,South Sudan,2020,12.21,3.64,0.01,3.47,NaN,20.199,223535.20
1665,South Sudan,2021,11.87,3.66,0.00,4.71,NaN,20.514,237110.27
1666,South Sudan,2022,12.07,3.89,0.00,4.77,NaN,20.846,248472.56


We can safely delete these data since there is no big impact on the study

In [11]:
# Delete Nan rows 
master_file =master_file[~master_file['GDP_per_capita'].isna()]
print(master_file.isnull().sum()  )

Country               0
Year                  0
Bovine Meat           0
Mutton & Goat Meat    0
Pigmeat               0
Poultry Meat          0
GDP_per_capita        0
Urban_population      0
Production            0
dtype: int64


In [14]:
save_path = 'data/processed/'
if not os.path.exists(save_path):
    os.makedirs(save_path)  
master_file.to_csv(os.path.join(save_path, 'meat_processed_merged_cleaned_data.csv'), index=False)